# MotionJSON Local UI with hosted and local SAM setup

This notebook launches the MotionJSON Local UI inside a Google Colab runtime and opens `/ui/` through Colab's built-in notebook port proxy. It is for connecting real SAM providers from the UI: local SAM2, local SAM3, Replicate SAM2 video, Roboflow SAM3, Fal SAM3 image, or custom SAM2/SAM3-compatible endpoints.

No public tunnel is started. Hosted provider keys are read from Colab userdata when available, with interactive fallback. You can also leave them blank and paste temporary credentials into the UI Model Connections form. Do not save private videos, provider credentials, or shared notebook outputs containing secrets.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/ptse8204/json-animated-video.git"
workdir = Path("/content/json-animated-video")

if not workdir.exists():
    subprocess.run(["git", "clone", repo_url, str(workdir)], check=True)
else:
    subprocess.run(["git", "-C", str(workdir), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=True)

os.chdir(workdir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[ui,hosted-segmentation,hosted-sam3,hosted-sam-vendors]"],
    check=True,
)


In [ ]:
print("Python:", sys.version.split()[0])
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("For local SAM2/SAM3, switch Colab to a GPU runtime before installing model packages.")
except Exception as exc:
    print("PyTorch is not importable yet:", type(exc).__name__)
    print("Hosted providers can still be linked. Local SAM setup needs torch plus the official SAM package.")


In [ ]:
from getpass import getpass

def colab_user_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return (value or "").strip()

for env_name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY", "HF_TOKEN"]:
    value = colab_user_secret(env_name)
    if not value:
        value = getpass(f"{env_name} (leave blank to skip): ").strip()
    if value:
        os.environ[env_name] = value

sam3_path = colab_user_secret("SAM3_LOCAL_MODEL")
if not sam3_path:
    sam3_path = input("SAM3_LOCAL_MODEL path (leave blank to skip): ").strip()
if sam3_path:
    os.environ["SAM3_LOCAL_MODEL"] = sam3_path
if os.environ.get("HF_TOKEN"):
    os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])

configured_names = [
    name
    for name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY", "HF_TOKEN", "SAM3_LOCAL_MODEL"]
    if os.environ.get(name)
]
print("Configured secret/path names:", configured_names or "none")


## Optional local SAM2 setup

Use local SAM2 for `Trace one object` when you want point/box-prompted video segmentation inside this Colab runtime. Run this only after selecting a GPU runtime. The official install and checkpoint commands are shown in the UI Model Connections panel too.

In [ ]:
RUN_LOCAL_SAM2_SETUP = False

if RUN_LOCAL_SAM2_SETUP:
    sam2_dir = Path("/content/sam2")
    if not sam2_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam2.git", str(sam2_dir)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(sam2_dir)], check=True)
    ckpt_dir = sam2_dir / "checkpoints"
    subprocess.run(["bash", "download_ckpts.sh"], cwd=str(ckpt_dir), check=True)
    os.environ.setdefault("SAM2_LOCAL_CHECKPOINT", str(ckpt_dir / "sam2.1_hiera_large.pt"))
    os.environ.setdefault("SAM2_LOCAL_CONFIG", "configs/sam2.1/sam2.1_hiera_l.yaml")
    os.environ.setdefault("SAM2_LOCAL_DEVICE", "cuda")
else:
    print("Set RUN_LOCAL_SAM2_SETUP = True to install official SAM2 and download checkpoints in this runtime.")
    print("Then use Model Connections -> SAM2 local and diagnose the saved checkpoint/config paths.")


## Optional local SAM3 setup

Use local SAM3 for concept prompts such as `red ball` or `person in white`. SAM3 local setup expects official package/model access and may require a Python/CUDA combination that differs from the default Colab image. If local SAM3 is not ready, use Roboflow SAM3 or Fal SAM3 image from Model Connections.

In [ ]:
RUN_LOCAL_SAM3_SETUP = False

if RUN_LOCAL_SAM3_SETUP:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("HF_TOKEN is required after Hugging Face access is approved for facebook/sam3.")
    sam3_dir = Path("/content/sam3")
    if not sam3_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam3.git", str(sam3_dir)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(sam3_dir)], check=True)
    os.environ.setdefault("SAM3_LOCAL_DEVICE", "cuda")
    print("Set SAM3_LOCAL_MODEL to the downloaded or cached facebook/sam3 model path before diagnosing in the UI.")
else:
    print("Set RUN_LOCAL_SAM3_SETUP = True only after confirming Colab Python/CUDA compatibility and Hugging Face access.")
    print("Hosted Roboflow SAM3 or Fal SAM3 image are usually faster to link for a first Colab UI run.")


In [ ]:
subprocess.run([sys.executable, "examples/make_demo_video.py", "--out", "examples/demo_red_ball.mp4"], check=True)
subprocess.run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"], check=True)
print("Demo video path to register in the UI: examples/demo_red_ball.mp4")
print("In the UI, open Model Connections, select a recommended SAM provider, diagnose setup, then validate the run config.")


In [ ]:
import time
from google.colab import output

port = 8766
ui_proc = subprocess.Popen(
    ["motionjson", "ui", "--no-open", "--host", "127.0.0.1", "--port", str(port)],
    cwd=str(workdir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(5)
print("MotionJSON UI started with: motionjson ui --no-open --host 127.0.0.1 --port", port)
print("Use Model Connections to link Replicate SAM2 video, Roboflow SAM3, Fal SAM3 image, local SAM2, local SAM3, or a custom endpoint.")
output.serve_kernel_port_as_iframe(port, path="/ui/", height=900)


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8766, path="/ui/")


In [ ]:
if ui_proc.poll() is None and ui_proc.stdout is not None:
    print("UI server is still running. Recent logs will appear here only after the process writes more output.")
else:
    print("UI server exited with code", ui_proc.returncode)


In [ ]:
if 'ui_proc' in globals() and ui_proc.poll() is None:
    ui_proc.terminate()
    print("MotionJSON UI stopped.")
